In [17]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, make_scorer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
import joblib

# 1. Load and Clean ADASYN Data
adasyn_df = pd.read_csv("adasyn_balanced_1.5M.csv")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour']

# Fill NaN in targets with 0 (disease absent)
adasyn_df[targets] = adasyn_df[targets].fillna(0)

# Verify no NaN remains
print("NaN counts after cleaning:")
print(adasyn_df[targets].isna().sum())

# 2. Feature Engineering
def create_features(X):
    X = X.copy()
    # Cyclical time features
    X['hour_sin'] = np.sin(2 * np.pi * X['hour']/24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour']/24)
    # Behavioral ratios
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour'])

# 3. Prepare Data
X = adasyn_df[features]
y = adasyn_df[targets]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Custom Multi-output Scorer
def multioutput_f1(y_true, y_pred):
    scores = []
    for i in range(y_true.shape[1]):
        report = classification_report(y_true.iloc[:,i], y_pred[:,i], 
                                     output_dict=True, zero_division=0)
        scores.append(report['weighted avg']['f1-score'])
    return np.mean(scores)

scorer = make_scorer(multioutput_f1)

# 5. Pipeline with Feature Engineering
pipeline = Pipeline([
    ('features', FunctionTransformer(create_features)),
    ('clf', MultiOutputClassifier(RandomForestClassifier(random_state=42)))
])

# 6. Hyperparameter Tuning (Reduced for Efficiency)
param_grid = {
    'clf__estimator__n_estimators': [100, 150],
    'clf__estimator__max_depth': [10, 20],
    'clf__estimator__min_samples_split': [2, 5],
    'clf__estimator__class_weight': ['balanced']
}

# 7. Cross-Validation Setup
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# 8. Grid Search with Reduced Data Sample (for speed)
tune_sample = 100000  # Use 100K samples for tuning
if len(X_train) > tune_sample:
    X_tune = X_train.sample(tune_sample, random_state=42)
    y_tune = y_train.loc[X_tune.index]
else:
    X_tune, y_tune = X_train, y_train

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring=scorer,
    verbose=2,
    n_jobs=1  # Reduce memory usage
)

# 9. Train Model
print("Starting hyperparameter tuning...")
grid_search.fit(X_tune, y_tune)

# 10. Final Model Training
best_model = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)

print("\nTraining final model on full data...")
best_model.fit(X_train, y_train)

# 11. Evaluation
y_pred = best_model.predict(X_test)
print("\nTest Performance:")
for i, target in enumerate(targets):
    print(f"\n{target}:")
    print(classification_report(y_test.iloc[:,i], y_pred[:,i]))

# 12. Save Model
joblib.dump(best_model, 'random_forest_oversampled_tuned.pkl')
print("\nModel saved as random_forest_oversampled_tuned.pkl")

NaN counts after cleaning:
oestrus     0
calving     0
lameness    0
mastitis    0
dtype: int64
Starting hyperparameter tuning...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END clf__estimator__class_weight=balanced, clf__estimator__max_depth=10, clf__estimator__min_samples_split=2, clf__estimator__n_estimators=100; total time=  54.6s
[CV] END clf__estimator__class_weight=balanced, clf__estimator__max_depth=10, clf__estimator__min_samples_split=2, clf__estimator__n_estimators=100; total time=  53.5s
[CV] END clf__estimator__class_weight=balanced, clf__estimator__max_depth=10, clf__estimator__min_samples_split=2, clf__estimator__n_estimators=100; total time=  52.3s
[CV] END clf__estimator__class_weight=balanced, clf__estimator__max_depth=10, clf__estimator__min_samples_split=2, clf__estimator__n_estimators=150; total time= 1.4min
[CV] END clf__estimator__class_weight=balanced, clf__estimator__max_depth=10, clf__estimator__min_samples_split=2, clf__estimator__n_estim